# Spreading from a single active site: DP or BVH infinite noise?

This notebook reads the output of

* `../generators/get_upper_lower_binary_spreading_bvh_b1.jl` (strong temporal disorder, `block_len = 1`),
* `../generators/get_upper_lower_binary_spreading_bvh_b6.jl` (same, `block_len = 6`),
* `../generators/get_upper_lower_binary_spreading_clean.jl` (clean Stavskaya, the DP control),

and asks one question: **as $t$ grows, do the spreading observables behave like power laws (a conventional
critical point, e.g. DP), or like the logarithmic laws of the infinite-noise critical point of
Barghathi, Vojta & Hoyos** (BVH, *Contact process with temporal disorder*, PRE 94, 022111 (2016),
arXiv:1603.08075, Sec. IV and Figs. 4–7)?

How to use it: set the parameter cell to the knobs of the generator you ran, run everything, and read
the markdown above each figure. It says what the figure shows, what DP predicts, what BVH predicts,
and what to watch out for. The last section prints a short verdict table.

**Contents**
1. The model and the spreading experiment
2. What the generator stores, and the observables built from it
3. Theory: what DP and BVH predict
4. Parameters and loading (plus sanity checks)
5. Survival probability, $1/P_s$ vs $\ln t$ (BVH Fig. 4)
6. Running exponent $\delta_{\rm eff}$ (BVH Fig. 5)
7. Number of active sites and cluster radius (BVH Fig. 6)
8. Fits over one window
9. Crossing times off criticality (BVH Fig. 7)
10. Verdict table

## 1. The model and the spreading experiment

**Stavskaya rule.** Sites $i\in\mathbb Z$ carry $\eta_i(t)\in\{0,1\}$; $\eta=1$ is *healthy*, $\eta=0$ is *active*.
At every step, independently for every site,
$$
\eta_i(t+1)=\begin{cases}1 & \text{with probability } \varepsilon(t),\\[2pt]
\eta_{i-1}(t)\,\eta_i(t) & \text{otherwise.}\end{cases}
$$
So an active site stays active and infects its right neighbour, unless the "healing" coin (probability
$\varepsilon$) fires. The all-healthy state is absorbing. Clean model: $\varepsilon(t)=\varepsilon$, with a DP
transition at $\varepsilon^*=0.29450(5)$ (Mendonça, arXiv:1011.1489).

**Temporal disorder (upper/lower binary, block length $b$).** At the start of every block of $b$ steps a new
$$
\varepsilon=\begin{cases}\varepsilon_u & \text{with probability } p \quad(\text{inactive step, } \varepsilon_u>\varepsilon^*)\\
\varepsilon_l=\varepsilon_u/20 & \text{with probability } 1-p \quad(\text{active step})\end{cases}
$$
is drawn and held for $b$ steps. The generators label a parameter set by the mean
$\bar\varepsilon=p\,\varepsilon_u+(1-p)\,\varepsilon_l=\big(p+\tfrac{1-p}{20}\big)\varepsilon_u$
(for $p=0.2$: $\bar\varepsilon=0.24\,\varepsilon_u$). This mirrors BVH's 1D runs: the active rate with probability 0.8,
the inactive one 20 times smaller, and $\Delta t=b$. A pilot scan put the critical point near
$\varepsilon_u\approx0.60$ ($b=1$) and $\approx0.46$ ($b=6$).

**Spreading experiment.** At $t=0$ one site $x_0$ is active and all others are healthy. Each run gets its own
disorder sequence $\varepsilon(t)$: *one run per disorder realization*, as in BVH. Averages over runs therefore
average over the intrinsic noise and over the disorder at once.

**Light cone.** The rule only looks at $i-1$ and $i$, so a cluster that starts at $x_0$ stays inside
$$x_0\le i\le x_0+t .$$
The cone's axis is $x_0+t/2$. In the usual directed-percolation coordinates $x'=2i-t$ the cone is symmetric,
$|x'-2x_0|\le t$, so distances are measured from $x_0+t/2$. Because nothing can leave the cone, the code
simulates the **infinite** chain exactly: there are no finite-size effects, and each step only updates the
sites between the leftmost active site and one past the rightmost one.

## 2. What the generator stores, and the observables

Runs come in chunks of $R_c$ = `runs_per_chunk` runs. Each chunk file holds, at every output time $t$
(log spaced, `points_per_decade` per decade), sums over its runs $r$. With $A_r(t)$ the set of active sites of run
$r$ and $N_r(t)=|A_r(t)|$:

| column | meaning |
|---|---|
| `runs` | $R_c$ |
| `surv` | $\sum_r \mathbf 1[N_r(t)>0]$, the number of surviving runs |
| `sum_n`, `sum_n2` | $\sum_r N_r(t)$ and $\sum_r N_r(t)^2$ (dead runs count 0) |
| `sum_x2` | $\sum_r\sum_{i\in A_r(t)}\big(i-x_0-\tfrac t2\big)^2$ |
| `sum_r2`, `sum_r2sq` | sums over surviving runs of $R_r^2=\frac{1}{N_r}\sum_{i\in A_r}(i-x_0-\frac t2)^2$ and of $R_r^4$ |

Summing over all chunks ($R$ runs in total) gives the three observables BVH use:
$$
P_s(t)=\frac{1}{R}\sum_r \mathbf 1[N_r(t)>0],\qquad
N(t)=\frac1R\sum_r N_r(t),\qquad
R^2(t)=\frac{\sum_r\sum_{i\in A_r}(i-x_0-t/2)^2}{\sum_r N_r(t)} .
$$
* $P_s$ is the **survival probability**.
* $N$ is the **mean number of active sites**, averaged over *all* runs (dead ones count 0); $N/P_s$ is the mean size
  of a surviving cluster.
* $R$ is the **mean-square spread** of the active sites about the cone axis, weighted by sites (the Grassberger
  convention). It obeys $R\le t/2$.

**Error bars** come from a bootstrap over chunks: resample the chunks with replacement and recompute the
observables. The chunks are statistically independent (each run has its own disorder), so this captures both
the intrinsic noise and the disorder fluctuations, which dominate at strong disorder.

## 3. Theory: what the two pictures predict

### Clean DP (a conventional critical point)
At the critical point everything is a power law:
$$P_s(t)\sim t^{-\delta},\qquad N(t)\sim t^{\theta},\qquad R^2(t)\sim t^{2/z},$$
with 1+1D values $\delta=\beta/\nu_\parallel=0.1595$, $\theta=0.3137$, $z=\nu_\parallel/\nu_\perp=1.5807$
($\nu_\parallel=1.7338$), linked by the hyperscaling relation $\theta=d/z-2\delta$ ($d=1$).
Off criticality, with $\Delta=(\varepsilon-\varepsilon^*)/\varepsilon^*$,
$P_s(t,\Delta)=t^{-\delta}\,f\!\big(\Delta\, t^{1/\nu_\parallel}\big)$: curves leave the critical one at $t_x\sim|\Delta|^{-\nu_\parallel}$.

### Why temporal disorder matters (Harris/Kinzel criterion)
Disorder that is random in time but uniform in space is relevant at a critical point if $\nu_\parallel<2$.
The argument: over a time window $t$ the averaged control parameter fluctuates by $\sigma\sqrt{b/t}$, while the
distance from criticality that the system can resolve at time $t$ is $t^{-1/\nu_\parallel}$. The ratio
$\sigma\sqrt b\,t^{1/\nu_\parallel-1/2}$ grows when $\nu_\parallel<2$. For DP $\nu_\parallel=1.73<2$: temporal
disorder is relevant, but only barely, so the crossover away from DP is slow. It sets in around
$$t_\times\sim(\sigma^2 b)^{-\nu_\parallel/(2-\nu_\parallel)}\approx(\sigma^2b)^{-6.5},$$
which is why longer blocks $b$ (and larger contrast) show the disordered behaviour at earlier times.

### BVH: the infinite-noise critical point
BVH's strong-noise renormalization group finds that the disorder does not settle at a finite strength: the
effective noise grows without bound, and time scales enter only through $\ln t$. At criticality (their
Eqs. 23–25, $z=1$ with logarithmic corrections):
$$
P_s(t)\sim(\ln t)^{-\bar\delta},\ \ \bar\delta=1;\qquad
N(t)\sim t\,(\ln t)^{-y_N};\qquad
R(t)\sim t\,(\ln t)^{-y_R}.
$$
For the 1D contact process they found $y_N=3.6$ and $y_R=1.7$, the same for $\Delta t=6$ and $3$. The *form* is
the universal prediction; the $y$ values here are fitted rather than assumed. Useful linearizations (BVH plot
exactly these):
$$
\frac{1}{P_s}=a+B\ln t,\qquad \Big(\frac{N}{t}\Big)^{-1/y_N}=a_N+b_N\ln t,\qquad \Big(\frac{R}{t}\Big)^{-1/y_R}=a_R+b_R\ln t .
$$
Off criticality the scaling variable is $r\,(\ln t)^2$ ($\bar\nu=1/2$ in the variable $\ln t$), so an off-critical
curve leaves the critical one at
$$\ln t_x\propto r^{-1/2}\qquad(\text{vs. } \ln t_x\approx\nu_\parallel\ln(1/r)\ \text{for a power law}).$$
A physical picture: rare long stretches of active steps let a cluster spread ballistically (speed $\le1$ site per
step here), and rare long stretches of inactive steps kill most clusters. The history of such rare periods, not
the average, controls the long-time behaviour, hence $z=1$ and logarithms.

### What this means for *finite* times (important when reading the plots)
The local exponents approach their BVH limits only logarithmically:
$$
\delta_{\rm eff}=-\frac{d\ln P_s}{d\ln t}=\frac{\bar\delta}{\ln t}\to0,\qquad
\theta_{\rm eff}=\frac{d\ln N}{d\ln t}\simeq1-\frac{y_N}{\ln t},\qquad
\frac1{z_{\rm eff}}=\frac{d\ln R}{d\ln t}\simeq1-\frac{y_R}{\ln t}.
$$
At $t=10^5$ ($\ln t=11.5$) and with BVH's $y$ values this gives $\theta_{\rm eff}\approx0.69$ and
$1/z_{\rm eff}\approx0.85$, far from DP's $0.314$ and $0.633$. So $N$ and $R$ are strong discriminators even
though $P_s$ decays so slowly. The plots below draw these asymptotic BVH curves as dotted lines. They only hold
once $\ln t\gg y$, so expect them to be approached from below.

## 4. Parameters and loading
Set these to the knobs of the generator you ran. The defaults are those of
`get_upper_lower_binary_spreading_bvh_b1.jl`. The parameter sets are rebuilt with exactly the generator's
arithmetic (`round(x, digits=6)` in Julia ↔ `round(x, 6)` here), so the file names match.

| generator | `AVG_EPS_C` | `AVG_EPS_RATE` | `P_VAL` | `BLOCK_LEN` |
|---|---|---|---|---|
| `..._spreading_bvh_b1.jl` | 0.144 | 0.0012 | 0.2 | 1 |
| `..._spreading_bvh_b6.jl` | 0.1104 | 0.0012 | 0.2 | 6 |
| `..._spreading_clean.jl` | 0.2945 | 0.0001 | 1.0 | 1 |

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))          # run from stavskya_mc/block_disorder/analysis
import time_log_tools as tl                  # fit_power_law, fit_bvh, inverse_activity_slope, crossing_time
import spreading_tools as st                 # load_spreading, local_slope, best_log_exponent

plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.2, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 13})

In [ ]:
# ---- mirror the knobs of the generator you ran: get_upper_lower_binary_spreading_fine_b6.jl (2026-10-05 fine-grid copy)
DATA_ROOT = "../../data/spreading"   # small (~10 MB per generator), so kept in the repo (git-ignored); see ../ssh_transfer.txt
AVG_EPS_C = 0.110784               # average_epsilon_c    (bvh_b6: 0.1104, clean: 0.2945)
AVG_EPS_RATE = 0.000216           # average_epsilon_rate (clean: 0.0001)
P_VAL = 0.2                     # p_val                (clean: 1.0)
LOWER_DIV = 20
EPS_STEPS = list(range(-4, 5))   # FINE grid: 9 values
BLOCK_LEN = 6                   # (bvh_b6: 6)
T_MAX = 100000
PPD = 20
NUM_CHUNKS = 200
RUNS_PER_CHUNK = 500
CHUNK_OFFSET = 0                # load chunks CHUNK_OFFSET+1 ... CHUNK_OFFSET+NUM_CHUNKS
CONTROL_C = None                # your current critical estimate of epsilon_bar; None -> middle value
CLEAN_EPS = 0.2945              # overlay the clean run (spreading_clean.jl) at this epsilon; None = no overlay
FIT_TMIN = 1e2                  # window for fits and exponent scans; FIT_TMAX None -> T_MAX
FIT_TMAX = None
Y_N = 3.6                       # BVH's 1D contact-process values, used for the reference curves
Y_R = 1.7
SPAN = 2.0                      # local slopes are measured over a factor SPAN in time
N_BOOT = 200                    # bootstrap resamples of the chunks
FIG_DIR = "figs"

In [ ]:
MODEL_DIR = "time_rand_window_binary"
f = P_VAL + (1 - P_VAL) / LOWER_DIV
sets = []                                    # (epsilon_bar, epsilon_u, epsilon_l, p)
for i in EPS_STEPS:
    u = round(AVG_EPS_C / f + i * (AVG_EPS_RATE / f), 6)
    l = round(u / LOWER_DIV, 6)
    sets.append((round(P_VAL * u + (1 - P_VAL) * l, 6), u, l, P_VAL))
controls = [s[0] for s in sets]
CONTROL_C = CONTROL_C if CONTROL_C is not None else controls[len(controls) // 2]
CONTROL_LABEL = r"$\bar\varepsilon$"
FIT_TMAX = FIT_TMAX or T_MAX
Path(FIG_DIR).mkdir(exist_ok=True)

def load(u, l, p, block_len):
    return st.load_spreading(DATA_ROOT, MODEL_DIR, T_MAX, u, l, p, block_len, PPD, RUNS_PER_CHUNK,
                             NUM_CHUNKS, chunk_offset=CHUNK_OFFSET)

runs, obs, se = {}, {}, {}
for c, u, l, p in sets:
    try:
        runs[c] = load(u, l, p, BLOCK_LEN)
    except FileNotFoundError as e:
        print(f"skipping {c}: {e}")
        continue
    obs[c] = runs[c].observables()
    se[c] = runs[c].bootstrap_se(n_boot=N_BOOT)
controls = [c for c in controls if c in runs]
if CONTROL_C not in runs:
    CONTROL_C = controls[len(controls) // 2]
t = runs[controls[0]].times

clean = None
if CLEAN_EPS is not None:
    try:
        clean = load(round(CLEAN_EPS, 6), round(CLEAN_EPS / LOWER_DIV, 6), 1.0, 1).observables()
    except FileNotFoundError as e:
        print(f"no clean overlay: {e}")

cmap = plt.colormaps["viridis"].resampled(len(controls))
colors = {c: cmap(i) for i, c in enumerate(controls)}
eps_u_of = {c: u for c, u, l, p in sets}
pd.DataFrame({c: {"epsilon_u": eps_u_of[c], "chunks": runs[c].n_chunks, "runs": runs[c].n_runs,
                  "P_s(t_max)": obs[c]["P_s"][-1], "N(t_max)": obs[c]["N"][-1],
                  "R(t_max)": np.sqrt(obs[c]["R2"][-1])} for c in controls}).T

**Sanity checks.** Every run starts with exactly one active site, so $P_s(0)=1$, $N(0)=1$ and $R^2(0)=0$.
The light cone bounds the spread, $R(t)\le t/2$. A failure here means the files do not come from the generator
you think they do.

In [ ]:
for c in controls:
    o = obs[c]
    ok = (abs(o["P_s"][0] - 1) < 1e-12 and abs(o["N"][0] - 1) < 1e-12 and abs(o["R2"][0]) < 1e-12
          and np.all(np.nan_to_num(np.sqrt(o["R2"][1:])) <= t[1:] / 2 + 1e-9)
          and np.all(np.diff(o["P_s"]) <= 1e-12))
    print(f"eps_bar = {c}: {'OK' if ok else 'PROBLEM'}  ({runs[c].n_runs} runs in {runs[c].n_chunks} chunks)")

## 5. Survival probability: $1/P_s$ against $\ln t$ (BVH Fig. 4)

**What is plotted.** Left: $1/P_s$ against $\ln t$ for every $\bar\varepsilon$ (with bootstrap error bars; the
error of $1/P_s$ is $\sigma_{P_s}/P_s^2$). Right: the local slope
$$B_{\rm eff}(t)=\frac{1/P_s(t)-1/P_s(t/s)}{\ln s},\qquad s=\texttt{SPAN},$$
i.e. $d(1/P_s)/d\ln t$ measured over a factor $s$ in time.

**How to read it.**
* **BVH critical point:** $1/P_s=a+B\ln t$, a *straight line*, so $B_{\rm eff}$ is *flat*. On the inactive side
  (larger $\bar\varepsilon$) curves bend upward ($B_{\rm eff}$ grows); on the active side they bend down and
  flatten ($B_{\rm eff}\to0$, since $P_s$ tends to a constant).
* **Power-law critical point:** $1/P_s=t^{\delta}$ curves *upward* even at criticality, and $B_{\rm eff}$ grows
  like $\delta\,t^{\delta}$. The dashed clean-Stavskaya curve shows what that looks like.
* So the critical $\bar\varepsilon$ is the one whose $B_{\rm eff}$ stays flat the longest. Early times
  ($\ln t\lesssim3$) are non-universal and should be ignored.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for c in controls:
    o, e = obs[c], se[c]
    m = (t > 0) & (o["P_s"] > 0)
    axs[0].errorbar(np.log(t[m]), 1 / o["P_s"][m], yerr=e["P_s"][m] / o["P_s"][m] ** 2,
                    color=colors[c], label=f"{c}", lw=1)
    ts, s = tl.inverse_activity_slope(t, o["P_s"], span=SPAN)
    axs[1].plot(np.log(ts), s, color=colors[c])
if clean is not None:
    m = (t > 0) & (clean["P_s"] > 0)
    axs[0].plot(np.log(t[m]), 1 / clean["P_s"][m], "k--", lw=1, label=f"clean {CLEAN_EPS}")
    ts, s = tl.inverse_activity_slope(t, clean["P_s"], span=SPAN)
    axs[1].plot(np.log(ts), s, "k--", lw=1)
axs[0].set_xlabel(r"$\ln t$"); axs[0].set_ylabel(r"$1/P_s$   (BVH: straight line)")
axs[1].set_xlabel(r"$\ln t$"); axs[1].set_ylabel(r"$B_{\rm eff}=d(1/P_s)/d\ln t$   (BVH: flat)")
axs[0].legend(title=CONTROL_LABEL, fontsize=8)
fig.suptitle(f"spreading, p={P_VAL}, block_len={BLOCK_LEN}")
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/spreading_p{P_VAL}_bl{BLOCK_LEN}_inverse_Ps.png", dpi=150)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for c in controls:
    if c > 0.1116:
        continue
    o, e = obs[c], se[c]
    m = (t > 0) & (o["P_s"] > 0)
    axs[0].errorbar(np.log(t[m]), 1 / o["P_s"][m], yerr=e["P_s"][m] / o["P_s"][m] ** 2,
                    color=colors[c], label=f"{c}", lw=1)
    ts, s = tl.inverse_activity_slope(t, o["P_s"], span=SPAN)
    axs[1].plot(np.log(ts), s, color=colors[c])
if clean is not None:
    m = (t > 0) & (clean["P_s"] > 0)
    axs[0].plot(np.log(t[m]), 1 / clean["P_s"][m], "k--", lw=1, label=f"clean {CLEAN_EPS}")
    ts, s = tl.inverse_activity_slope(t, clean["P_s"], span=SPAN)
    axs[1].plot(np.log(ts), s, "k--", lw=1)
axs[0].set_xlabel(r"$\ln t$"); axs[0].set_ylabel(r"$1/P_s$   (BVH: straight line)")
axs[1].set_xlabel(r"$\ln t$"); axs[1].set_ylabel(r"$B_{\rm eff}=d(1/P_s)/d\ln t$   (BVH: flat)")
axs[0].legend(title=CONTROL_LABEL, fontsize=8)
fig.suptitle(f"spreading, p={P_VAL}, block_len={BLOCK_LEN}")
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/spreading_p{P_VAL}_bl{BLOCK_LEN}_inverse_Ps.png", dpi=150)

## 6. Running exponent $\delta_{\rm eff}$ (BVH Fig. 5)

**What is plotted.** Left: $\log_{10}P_s$ against $\log_{10}t$ (a power law is a straight line). Right: the
inverse of the running exponent
$$\delta_{\rm eff}(t)=-\frac{\ln P_s(t)-\ln P_s(t/s)}{\ln s}$$
against $\ln t$.

**How to read it.**
* **Power law** $P_s\sim t^{-\delta}$: $\delta_{\rm eff}\to\delta$, so $1/\delta_{\rm eff}$ goes *flat*
  (clean DP: $1/0.1595=6.3$, grey line; the dashed clean curve should settle there).
* **BVH** $P_s=1/(a+B\ln t)$: then exactly $\delta_{\rm eff}=B/(a+B\ln t)$, i.e.
  $$\frac1{\delta_{\rm eff}}=\ln t+\frac aB ,$$
  a straight line of **slope 1** in $\ln t$ (slope $1/\bar\delta$ for $P_s\sim(\ln t)^{-\bar\delta}$). The dotted
  guide has slope 1 and passes through the critical curve at the middle of the fit window. BVH Fig. 5 (inset)
  shows exactly this.
* Off criticality: inactive-side curves first follow the critical one, then turn *down* ($\delta_{\rm eff}$ grows,
  exponential death); active-side curves turn *up* ($\delta_{\rm eff}\to0$, $P_s\to$ const). A curve that follows
  the slope-1 line over several units of $\ln t$ is the signature to look for. Noise blows up where
  $\delta_{\rm eff}\approx0$, so points with $\delta_{\rm eff}<0.01$ are dropped.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for c in controls:
    o = obs[c]; m = (t > 0) & (o["P_s"] > 0)
    axs[0].plot(np.log10(t[m]), np.log10(o["P_s"][m]), color=colors[c], label=f"{c}")
    T, s = st.local_slope(t, o["P_s"], span=SPAN)
    d = -s; ok = d > 0.01
    axs[1].plot(np.log(T[ok]), 1 / d[ok], color=colors[c])
    if c == CONTROL_C and ok.any():
        w = ok & (T >= FIT_TMIN) & (T <= FIT_TMAX)
        if w.any():
            x0 = np.log(np.sqrt(FIT_TMIN * FIT_TMAX)); y0 = np.interp(x0, np.log(T[ok]), 1 / d[ok])
            xx = np.log(T[ok]); axs[1].plot(xx, y0 + (xx - x0), ":", color="k", lw=1.2, label="BVH: slope 1")
if clean is not None:
    m = (t > 0) & (clean["P_s"] > 0)
    axs[0].plot(np.log10(t[m]), np.log10(clean["P_s"][m]), "k--", lw=1, label="clean")
    T, s = st.local_slope(t, clean["P_s"], span=SPAN); ok = -s > 0.01
    axs[1].plot(np.log(T[ok]), -1 / s[ok], "k--", lw=1, label="clean")
axs[1].axhline(1 / st.DP["delta"], color="grey", lw=0.8, label=r"clean DP $1/\delta=6.3$")
axs[0].set_xlabel(r"$\log_{10} t$"); axs[0].set_ylabel(r"$\log_{10} P_s$   (power law: straight)")
axs[1].set_xlabel(r"$\ln t$"); axs[1].set_ylabel(r"$1/\delta_{\rm eff}$   (DP: flat, BVH: slope 1)")
axs[1].set_ylim(0, 60); axs[1].legend(fontsize=9)
axs[0].legend(title=CONTROL_LABEL, fontsize=8)
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/spreading_p{P_VAL}_bl{BLOCK_LEN}_delta_eff.png", dpi=150)

## 7. Number of active sites and cluster radius (BVH Fig. 6), at $\bar\varepsilon=$ `CONTROL_C`

**What is plotted.**
* Top: BVH's linearizations $(N/t)^{-1/y}$ and $(R/t)^{-1/y}$ against $\ln t$, for BVH's contact-process
  exponents (`Y_N`, `Y_R`) and for the exponent that makes the curve *straightest* over the fit window. That is
  found by scanning $y$ and minimizing the relative rms deviation from a straight line (`best_log_exponent`).
* Bottom: the local exponents
  $$\theta_{\rm eff}=\frac{d\ln N}{d\ln t},\qquad \frac1{z_{\rm eff}}=\frac{d\ln R}{d\ln t},$$
  measured over a factor `SPAN` in time.

**How to read it.**
* **DP:** $\theta_{\rm eff}\to\theta=0.314$ and $1/z_{\rm eff}\to1/z=0.633$ (grey lines); the clean dashed curves
  should settle there.
* **BVH:** $N\sim t(\ln t)^{-y_N}$ and $R\sim t(\ln t)^{-y_R}$, so $\theta_{\rm eff}\simeq1-y_N/\ln t$ and
  $1/z_{\rm eff}\simeq1-y_R/\ln t$ (dotted curves): both creep toward **1** (ballistic spreading, $z=1$)
  from below. The crucial difference from DP is the *direction*: at a DP point they level off at 0.314 and 0.633,
  at BVH they keep **rising**.
* **The straightest $y$.** If it sits at the top of the scanned range (8), the data prefer $y\to\infty$, which is
  a pure power law ($g^{-1/y}\approx1-\ln g/y$ is linear in $\ln t$ only if $\ln g$ is). A finite best $y$ means
  logarithmic corrections. Do not expect BVH's 3.6 and 1.7 exactly: those are contact-process numbers.
* **Bound:** $R\le t/2$, so $1/z_{\rm eff}\le1$ at late times. Values above 1 are noise.

In [ ]:
o = obs[CONTROL_C]
m = t > 1
gN, gR = o["N"][m] / t[m], np.sqrt(o["R2"][m]) / t[m]
yN_best, tabN = st.best_log_exponent(t[m], gN, FIT_TMIN, FIT_TMAX)
yR_best, tabR = st.best_log_exponent(t[m], gR, FIT_TMIN, FIT_TMAX)

fig, axs = plt.subplots(2, 2, figsize=(13, 9))
for ax, g, y_bvh, y_best, name in ((axs[0, 0], gN, Y_N, yN_best, "N/t"), (axs[0, 1], gR, Y_R, yR_best, "R/t")):
    ax.plot(np.log(t[m]), g ** (-1 / y_bvh), "-", label=f"y = {y_bvh:.2f} (BVH contact process)")
    if np.isfinite(y_best):
        ax.plot(np.log(t[m]), g ** (-1 / y_best), "--", label=f"y = {y_best:.2f} (straightest)")
    ax.set_xlabel(r"$\ln t$"); ax.set_ylabel(f"({name})$^{{-1/y}}$   (BVH: straight)"); ax.legend(fontsize=9)

lt = np.linspace(np.log(max(FIT_TMIN, 10)), np.log(t[-1]), 100)
for ax, y_obs, dp, y_bvh, name in ((axs[1, 0], o["N"], st.DP["theta"], Y_N, r"$\theta_{\rm eff}=d\ln N/d\ln t$"),
                                   (axs[1, 1], np.sqrt(o["R2"]), 1 / st.DP["z"], Y_R, r"$1/z_{\rm eff}=d\ln R/d\ln t$")):
    T, s = st.local_slope(t, y_obs, span=SPAN)
    ax.plot(np.log(T), s, color=colors[CONTROL_C], label=f"{CONTROL_C}")
    ax.axhline(dp, color="grey", lw=0.8, label="clean DP")
    ax.plot(lt, 1 - y_bvh / lt, ":", color="k", lw=1.2, label=rf"BVH $1-{y_bvh}/\ln t$")
    ax.axhline(1, color="k", lw=0.5)
    ax.set_xlabel(r"$\ln t$"); ax.set_ylabel(name); ax.set_ylim(-0.1, 1.2)
if clean is not None:
    for ax, y_obs in ((axs[1, 0], clean["N"]), (axs[1, 1], np.sqrt(clean["R2"]))):
        T, s = st.local_slope(t, y_obs, span=SPAN)
        ax.plot(np.log(T), s, "k--", lw=1, label="clean")
for ax in axs[1]:
    ax.legend(fontsize=9)
fig.suptitle(f"p={P_VAL}, block_len={BLOCK_LEN}, " + CONTROL_LABEL + f" = {CONTROL_C}")
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/spreading_p{P_VAL}_bl{BLOCK_LEN}_N_R.png", dpi=150)
print(f"straightest exponents over [{FIT_TMIN:g}, {FIT_TMAX:g}]: y_N = {yN_best:.2f}, y_R = {yR_best:.2f}"
      "   (BVH 1D contact process: 3.6, 1.7; 8.0 = top of scan = power law preferred)")

## 8. Fits of $P_s$ over one window

Two weighted least-squares fits over `[FIT_TMIN, FIT_TMAX]`, with the bootstrap errors as weights:
* power law: $\ln P_s=c-\delta\ln t$;
* BVH: $1/P_s=a+B\ln t$.

**How to read it.** Compare the two $\chi^2_\nu$ within a row. The one that is clearly smaller wins; if they are
similar, the window is too short to discriminate. The same runs feed every time point, so neighbouring points
are strongly correlated and $\chi^2_\nu$ is *not* an absolute goodness of fit. Also watch the fitted power-law
$\delta$ as you move `FIT_TMIN` up: it should stay put at a power-law point and *shrink* under BVH.

In [ ]:
rows = []
for c in controls:
    oo, e = obs[c], se[c]
    fp = tl.fit_power_law(t, oo["P_s"], e["P_s"], FIT_TMIN, FIT_TMAX)
    fb = tl.fit_bvh(t, oo["P_s"], e["P_s"], FIT_TMIN, FIT_TMAX)
    rows.append({"eps_bar": c, "power: delta": fp["delta"], "power: chi2r": fp["chi2r"],
                 "BVH: B": fb["B"], "BVH: a/B = ln t0": fb["ln_t0"], "BVH: chi2r": fb["chi2r"], "points": fp["n"]})
print(f"fit window: {FIT_TMIN:g} <= t <= {FIT_TMAX:g}")
pd.DataFrame(rows).set_index("eps_bar").round(4)

## 9. Crossing times off criticality (BVH Fig. 7)

For each $\bar\varepsilon$ on the *inactive* side of `CONTROL_C`, with
$$r=\frac{\bar\varepsilon-\bar\varepsilon_c}{\bar\varepsilon_c},$$
$t_x$ is the first time $1/P_s$ exceeds $1.1\times$ the critical curve (BVH's criterion). The two pictures give
different straight lines:
* **BVH** (scaling variable $r(\ln t)^2$): $\ln t_x\propto r^{-1/2}$, so $r^{-1/2}$ against $\ln t_x$ is linear
  through the origin (up to a non-universal offset).
* **Power law** (scaling variable $r\,t^{1/\nu_\parallel}$): $t_x\propto r^{-\nu_\parallel}$, so $\ln t_x$ against
  $\ln(1/r)$ is linear with slope $\nu_\parallel$ (DP: 1.73).

**How to read it.** With only two or three inactive-side values this is a first look. Its value is that it does
not depend on $\delta$ at all. It is only meaningful once `CONTROL_C` is set to your best critical estimate;
curves that never cross within $t\le t_{\max}$ give no point.

In [ ]:
xs = []
for c in controls:
    r = (c - CONTROL_C) / CONTROL_C
    if r <= 0:
        continue
    xs.append({"eps_bar": c, "r": r, "t_x": tl.crossing_time(t, obs[CONTROL_C]["P_s"], obs[c]["P_s"], factor=1.1)})
cross = pd.DataFrame(xs)
display(cross)
ok = cross.dropna() if len(cross) else cross
if len(ok) >= 2:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
    axs[0].plot(np.log(ok.t_x), ok.r ** -0.5, "o-"); axs[0].set_xlabel(r"$\ln t_x$"); axs[0].set_ylabel(r"$r^{-1/2}$   (BVH: linear)")
    axs[1].plot(np.log(1 / ok.r), np.log(ok.t_x), "o-"); axs[1].set_xlabel(r"$\ln(1/r)$"); axs[1].set_ylabel(r"$\ln t_x$   (power law: slope $\nu_\parallel$)")
    fig.tight_layout(); fig.savefig(f"{FIG_DIR}/spreading_p{P_VAL}_bl{BLOCK_LEN}_crossing.png", dpi=150)
    print("power-law slope nu_par(eff) =", round(np.polyfit(np.log(1 / ok.r), np.log(ok.t_x), 1)[0], 3), "  (DP: 1.73)")
else:
    print("fewer than two inactive-side values crossed the critical curve; nothing to plot")

## 10. Verdict table at `CONTROL_C`

The diagnostics above in one table, evaluated at the latest times: averages of the local exponents over the last
decade $[t_{\max}/10,\,t_{\max}]$, and the slope of $1/\delta_{\rm eff}$ against $\ln t$ over the fit window.

| quantity | clean DP | BVH (asymptotic) | BVH at $t$ in the last decade |
|---|---|---|---|
| $\delta_{\rm eff}$ | 0.1595 | $\to0$ | $\approx1/\ln t$ |
| slope of $1/\delta_{\rm eff}$ vs $\ln t$ | 0 | 1 | 1 |
| $\theta_{\rm eff}$ | 0.314 | $\to1$ | $\approx1-y_N/\ln t$ |
| $1/z_{\rm eff}$ | 0.633 | $\to1$ | $\approx1-y_R/\ln t$ |

Read it together with the figures: one number can be pulled by noise, but a consistent pattern across all four
rows (and the clean column behaving like DP) is the result.

In [ ]:
def late_mean(y, frac=0.1):
    T, s = st.local_slope(t, y, span=SPAN)
    w = T >= T[-1] * frac
    return float(np.mean(s[w])) if w.any() else np.nan, float(np.mean(np.log(T[w]))) if w.any() else np.nan

def slope_inv_delta(Ps):
    T, s = st.local_slope(t, Ps, span=SPAN)
    w = (T >= FIT_TMIN) & (T <= FIT_TMAX) & (-s > 0.01)
    return float(np.polyfit(np.log(T[w]), -1 / s[w], 1)[0]) if w.sum() >= 3 else np.nan

rows = {}
for name, oo in [(f"eps_bar = {CONTROL_C}", obs[CONTROL_C])] + ([(f"clean {CLEAN_EPS}", clean)] if clean is not None else []):
    d, lnt = late_mean(oo["P_s"]); th, _ = late_mean(oo["N"]); iz, _ = late_mean(np.sqrt(oo["R2"]))
    rows[name] = {"delta_eff (last decade)": -d, "slope of 1/delta_eff vs ln t": slope_inv_delta(oo["P_s"]),
                  "theta_eff (last decade)": th, "1/z_eff (last decade)": iz}
lnt_late = np.log(t[-1]) - 0.5 * np.log(10)
rows["clean DP"] = {"delta_eff (last decade)": st.DP["delta"], "slope of 1/delta_eff vs ln t": 0.0,
                    "theta_eff (last decade)": st.DP["theta"], "1/z_eff (last decade)": 1 / st.DP["z"]}
rows[f"BVH at ln t = {lnt_late:.1f}"] = {"delta_eff (last decade)": 1 / lnt_late, "slope of 1/delta_eff vs ln t": 1.0,
                                        "theta_eff (last decade)": 1 - Y_N / lnt_late, "1/z_eff (last decade)": 1 - Y_R / lnt_late}
pd.DataFrame(rows).T.round(3)

## Joint test: is there ONE $\bar\varepsilon$ where $P_s$, $N$ and $R$ are all power laws? (added 2026-10-05)

For each observable, the slope of $\ln O$ against $\ln t$ on $[10^3,10^4]$ and on $[10^4,10^5]$, and the **drift** (late minus early).
Drift $=0$: a power law over both decades; $>0$: curving up (active side); $<0$: curving down (inactive side).

**Clean DP** (`analyze_spreading.ipynb`): the three drifts cross zero at the same $\varepsilon^*=0.2945$. **Here:** if they cross at clearly
different $\bar\varepsilon$, no single critical point makes this run look like a conventional (power-law) critical point, which is the
BVH-crossover signature. The coarse runs gave $P_s<N<R$ separated by about 0.5% of $\bar\varepsilon_c$ (H §7.19). The statistical errors of the
drifts are 0.001–0.005, small compared with the separations (bootstrap in `_claude_scratch/review_checks_2026_10_05.py`, section [10]).
Regime: critical region, $t\le10^5$; BVH asymptotics only once $\ln t\gg c$.

In [ ]:
# ---- joint test (2026-10-05): drift of the log-log slope between the decades [1e3,1e4] and [1e4,1e5]
def decade_slope(y, a, b):
    i, j = np.argmin(abs(t - a)), np.argmin(abs(t - b))
    with np.errstate(divide="ignore", invalid="ignore"):   # runs that died: P_s = 0
        return np.log(y[j] / y[i]) / np.log(t[j] / t[i])

def zero_cross(eps, d):                      # first eps where d goes from >= 0 to < 0 (linear between the two values)
    k = np.where((d[:-1] >= 0) & (d[1:] < 0))[0]
    return eps[k[0]] + (eps[k[0] + 1] - eps[k[0]]) * d[k[0]] / (d[k[0]] - d[k[0] + 1]) if len(k) else np.nan

drift = pd.DataFrame({c: {name: decade_slope(y, 1e4, 1e5) - decade_slope(y, 1e3, 1e4)
                          for name, y in (("P_s", obs[c]["P_s"]), ("N", obs[c]["N"]), ("R", np.sqrt(obs[c]["R2"])))}
                      for c in controls}).T
zero = {name: zero_cross(drift.index.values, drift[name].values) for name in drift}

ax = drift.plot(marker="o", figsize=(6.5, 4.2))
for (name, z), col in zip(zero.items(), ("C0", "C1", "C2")):
    ax.axvline(z, color=col, ls=":", lw=1)
ax.axhline(0, color="k", lw=0.8); ax.set_ylim(-0.2, 0.2)
ax.set_xlabel(CONTROL_LABEL); ax.set_ylabel(r"slope on [$10^4,10^5$] $-$ slope on [$10^3,10^4$]")
ax.set_title(f"joint test, p={P_VAL}, block_len={BLOCK_LEN}")
plt.savefig(f"{FIG_DIR}/spreading_p{P_VAL}_bl{BLOCK_LEN}_joint_test.png", dpi=150)
print("epsilon_bar where each observable is a power law over 1e3-1e5:", {k: round(float(v), 6) for k, v in zero.items()})
drift.round(4)